# Agentic FAQ Assistant

In [57]:
from openai import OpenAI
import requests
from minsearch import AppendableIndex
import json

In [58]:
openai_client = OpenAI()

### Data preprocessing

In [59]:
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name # add the course name to each document
        documents.append(doc)

In [60]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [61]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
    )

    return results

In [62]:
# define agent tool parameters
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [63]:
instructions = """
    You're a course teaching assistant. 
    You're given a question from a course student and your task is to answer it.
    If you want to look up the anser
""".strip()

tools = [search_tool]

question = 'I just discovered the course. Can I still join it?'

chat_messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

# send the initial prompt with little context and the tool option
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)

In [64]:
# the LLM decided to invoke the search tool
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Can I still join the course?"}', call_id='call_nyWIZco034jOri6gLQpI3iMJ', name='search', type='function_call', id='fc_0bee3cdeec7572bc00690b25da8204819fa1410f20b7647953', status='completed')

In [65]:
# add the tool call back to the context
call = response.output[0]
chat_messages.append(call)

In [66]:
# manually perform the tool call
args = {"query":"Can I still join the course?"}
search_results = search(*args)
search_results_json = json.dumps(search_results)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": search_results_json,
}

# add the output of tool to the context
chat_messages.append(call_output)

In [67]:
# new LLM prompt with enriched context
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)

In [68]:
response.output_text

"Yes, you can still join the course! If you haven't already, make sure to sign up and follow any enrollment procedures provided in the course's official page. If you have any specific questions or need help with the enrollment process, feel free to ask!"

### Function call + text output

In [69]:
instructions = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

If you want to look up the answer, explain why before making the call
""".strip()

In [70]:
tools = [search_tool]

question = 'I just discovered the course. Can I still join it?'

chat_messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)

In [71]:
len(response.output)

2

In [72]:
# the LLM produced both a text output and a function call
print(response.output[0])
print(response.output[1])

ResponseOutputMessage(id='msg_0da2f09c671f85bd00690b25dcfafc8192a829c8a49975b7b2', content=[ResponseOutputText(annotations=[], text='To provide you with the most accurate information about joining the course, I will check the FAQ database for any relevant details regarding enrollment deadlines and policies for late joiners. Let me look that up for you.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')
ResponseFunctionToolCall(arguments='{"query":"join course late enrollment"}', call_id='call_pOrrQICAtMRe4Fwb1r3jO1JR', name='search', type='function_call', id='fc_0da2f09c671f85bd00690b25de38208192a362af647da5d24a', status='completed')


In [73]:
def make_call(call):
    f_name = call.name
    arguments = json.loads(call.arguments)

    if f_name == 'search':
        results = search(**arguments)
    else:
        raise ValueError(f'unknown function {f_name}')

    json_results = json.dumps(results)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": json_results,
    }

### Agentic RAG loop

In [ ]:
question = 'I just discovered the course. Can I still join it?'

chat_messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

has_function_calls = True

while has_function_calls:
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=chat_messages,
        tools=tools
    )

    # add the LLM's response to the context history
    chat_messages.extend(response.output)
    
    for item in response.output:
        # print text outputs to the screen
        if item.type == 'message':
            print(item.content[0].text)
        
        if item.type == 'function_call':
            # add the tool call back to the context
            call_output = make_call(item)
            chat_messages.append(call_output)
            has_function_calls = True

To provide you with the most accurate information, I'll check the FAQ to see if there are policies regarding late enrollments or joining the course after it has started.
Yes, you can still join the course even after it has started. While there are deadlines for submitting final projects, you can submit homework and participate in the course activities. Just make sure to keep track of any upcoming deadlines!

If you have more specific questions about the enrollment process or any other details, feel free to ask!
